In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from noaa_coops import Station

from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

import utide

In [2]:
# Connect to NOAA station (Beaufort, NC)
duck = Station(id="8651370")

# Download 30 years of hourly water level data
df = duck.get_data(
    begin_date="19840101",
    end_date="20041231",
    product="hourly_height",
    datum="NAVD",
    units="metric",
    time_zone="gmt"
)

print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Total records: {len(df)}")

# The water level data is in column 'v'
df['water_level'] = pd.to_numeric(df['v'], errors='coerce')

# Fill small gaps
df['water_level'] = df['water_level'].interpolate(method='linear', limit=3)

print(f"Missing values after interpolation: {df['water_level'].isna().sum()}")

# Remove missing values
valid_mask = ~np.isnan(df['water_level'])
valid_time = df.index[valid_mask]
valid_water = df['water_level'][valid_mask].values

# Solve for tidal constituents
coef = utide.solve(
    valid_time,
    valid_water,
    lat= 36.1833,          # duck, NC latitude
    method='ols',
    conf_int='none',
    nodal=True,
    verbose=False
)

# Tidal predictions
tide = utide.reconstruct(valid_time, coef, verbose=False)
df_tide = pd.Series(tide['h'], index=valid_time, name='tidal_prediction')

# Calculate the non-tidal residual
df_residual = valid_water - tide['h']
df_residual = pd.Series(df_residual, index=valid_time, name='non_tidal_residual')

# Create plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# Tidal prediction (nA)
ax1.plot(df_tide.index, df_tide, 'r-', linewidth=0.8, label='Tidal Prediction (ηA)')
# ax1.plot(df_tide.index, valid_water, 'b-', linewidth=0.5, alpha=0.5, label='Observed')
ax1.set_ylabel('Water Level (m NAVD88)')
ax1.set_title('Tidal Prediction (ηA) (1992-2010)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Non-tidal residual (nNTR)
ax2.plot(df_residual.index, df_residual, 'g-', linewidth=0.5)
ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax2.set_ylabel('Non-Tidal Residual (m)')
ax2.set_xlabel('Date')
ax2.set_title('Non-Tidal Residual (ηNTR = Observed - Tidal Prediction)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Create a combined dataframe
results_df = pd.DataFrame({
    'datetime': df_tide.index,
    'tidal_prediction_etaA': df_tide.values,
    'non_tidal_residual_etaNTR': df_residual.values,
    'observed_water_level': valid_water
})

In [3]:
wis_filename = '/Users/rsahrae/PycharmProjects/PeaIsland_Hindcast/CASCADE/data/1984_2004_FOR_HANNAH.txt'
beach_slope = 0.004
berm_crest = 2 
g = 9.81

def load_wis_data_correct(filename):
    
    df_wis = pd.read_csv(filename, sep=r'\s+', header=None)
    
    print(f"DataFrame shape: {df_wis.shape}")
    print(f"Number of columns: {len(df_wis.columns)}")
    
    start_date = pd.Timestamp('1984-01-01 00:00:00')
    df_wis['datetime'] = [start_date + timedelta(hours=i) for i in range(len(df_wis))]
    df_wis.set_index('datetime', inplace=True)
    
    # Remove the first datetime column
    df_wis.drop(0, axis=1, inplace=True)
    
    print(f"Date range: {df_wis.index.min()} to {df_wis.index.max()}")
    print(f"Records: {len(df_wis)}")
    
    # Calculate statistics for each column
    col_stats = []
    for col in df_wis.columns:
        col_data = df_wis[col].dropna()
        if len(col_data) > 0:
            mean_val = col_data.mean()
            max_val = col_data.max()
            min_val = col_data.min()
            std_val = col_data.std()
            col_stats.append({
                'col': col,
                'mean': mean_val,
                'max': max_val,
                'min': min_val,
                'std': std_val
            })
    
    # Wave height
    hs_candidates = []
    for stat in col_stats:
        if 0.1 < stat['mean'] < 5 and 0 < stat['min'] and stat['max'] < 15:
            hs_candidates.append(stat)
    
    # Wave period
    tp_candidates = []
    for stat in col_stats:
        if 2 < stat['mean'] < 15 and 0 < stat['min'] and stat['max'] < 30:
            tp_candidates.append(stat)
    
    if hs_candidates:
        hs_candidates.sort(key=lambda x: abs(x['mean'] - 1.5))
        hs_col = hs_candidates[0]['col']
        print(f"\nIdentified wave height column: {hs_col} (mean={hs_candidates[0]['mean']:.2f} m)")
        df_wis['Hs'] = df_wis[hs_col]
    else:
        if 9 in df_wis.columns:
            print(f"\nUsing column 9 for wave height (mean={df_wis[9].mean():.2f} m)")
            df_wis['Hs'] = df_wis[9]
        else:
            print("Could not identify wave height column")
    
    if tp_candidates:
        tp_candidates.sort(key=lambda x: abs(x['mean'] - 8))
        tp_col = tp_candidates[0]['col']
        print(f"Identified wave period column: {tp_col} (mean={tp_candidates[0]['mean']:.2f} s)")
        df_wis['Tp'] = df_wis[tp_col]
    else:
        if 11 in df_wis.columns:
            print(f"Using column 11 for wave period (mean={df_wis[11].mean():.2f} s)")
            df_wis['Tp'] = df_wis[11]
        else:
            print("Could not identify wave period column")
    
    # Wave direction
    dir_candidates = []
    for stat in col_stats:
        if 0 < stat['mean'] < 360 and stat['max'] < 360:
            dir_candidates.append(stat)
    
    if dir_candidates:
        dir_col = dir_candidates[0]['col']
        print(f"Identified wave direction column: {dir_col} (mean={dir_candidates[0]['mean']:.1f}°)")
        df_wis['WAVD'] = df_wis[dir_col]
    elif 15 in df_wis.columns:
        df_wis['WAVD'] = df_wis[15]
    
    # Remove negative values
    if 'Hs' in df_wis.columns:
        df_wis['Hs'] = df_wis['Hs'].clip(lower=0)
    if 'Tp' in df_wis.columns:
        df_wis['Tp'] = df_wis['Tp'].clip(lower=0)
    
    print(f"\nWIS data loaded successfully!")
    print(f"Final columns: {df_wis.columns.tolist()}")
    
    return df_wis

In [4]:
# Load data from part A
# Set datetime index for tide data
if 'datetime' in results_df.columns:
    results_df['datetime'] = pd.to_datetime(results_df['datetime'])
    results_df.set_index('datetime', inplace=True)

# Load WIS data
df_wis = load_wis_data_correct(wis_filename)

# Create a merged dataframe with WIS timestamps
df_merged = pd.DataFrame(index=df_wis.index)

# Add WIS wave data
df_merged['Hs'] = df_wis['Hs']
df_merged['Tp'] = df_wis['Tp']
df_merged['WAVD'] = df_wis['WAVD']

# Add water level from tide data
if 'observed_water_level' in results_df.columns:
    water_level_col = 'observed_water_level'
elif 'water_level_clean' in results_df.columns:
    water_level_col = 'water_level_clean'
else:
    water_level_col = 'v'

# Align tide data to WIS timestamps
df_merged['water_level'] = results_df[water_level_col].reindex(df_wis.index).interpolate(
    method='linear', limit=24, limit_area='inside'
)

# Remove rows with missing data
df_merged = df_merged.dropna(subset=['Hs', 'Tp', 'water_level'])

print(f"Merged data shape: {df_merged.shape}")
print(f"Date range: {df_merged.index.min()} to {df_merged.index.max()}")
print(f"Wave height range: {df_merged['Hs'].min():.2f} to {df_merged['Hs'].max():.2f} m")
print(f"Wave period range: {df_merged['Tp'].min():.2f} to {df_merged['Tp'].max():.2f} s")

In [5]:
df_merged

In [6]:
def calculate_r2_percent(Hs, Tp, slope):
    """Calculate R2% (2% exceedance runup) using Stockdon et al. (2006)"""
    # Deep water wavelength
    L0 = (g * Tp**2) / (2 * np.pi)
    
    # Setup component
    setup = 0.35 * slope * np.sqrt(Hs * L0)
    
    # Swash components
    S_incident = 0.75 * slope * np.sqrt(Hs * L0)
    S_infragravity = 0.06 * np.sqrt(Hs * L0)
    S_total = np.sqrt(S_incident**2 + S_infragravity**2)
    
    # R2% runup
    R2 = 1.1 * (setup + S_total/2)
    
    return R2
    
df_merged['R2'] = calculate_r2_percent(df_merged['Hs'], df_merged['Tp'], beach_slope)
df_merged['TWL'] = df_merged['water_level'] + df_merged['R2']

In [7]:
df_merged

In [8]:
fig, axes = plt.subplots(4, 1, figsize=(15, 14))

# Plot Hs
axes[0].plot(df_merged.index, df_merged['Hs'], 'b-', linewidth=0.5, alpha=0.7)
axes[0].set_ylabel('Significant Wave Height (m)')
axes[0].set_title(f'Significant Wave Height (WIS Data) ({df_merged.index.min().year}-{df_merged.index.max().year})')
axes[0].grid(True, alpha=0.3)
if len(df_merged) > 0:
    axes[0].set_ylim([0, df_merged['Hs'].quantile(0.99)])

# Plot Tp
axes[1].plot(df_merged.index, df_merged['Tp'], 'r-', linewidth=0.5, alpha=0.7)
axes[1].set_ylabel('Peak Period (s)')
axes[1].set_title('Wave Period (Tp)')
axes[1].grid(True, alpha=0.3)
if len(df_merged) > 0:
    axes[1].set_ylim([0, df_merged['Tp'].quantile(0.99)])

# Plot R2%
axes[2].plot(df_merged.index, df_merged['R2'], 'g-', linewidth=0.5, alpha=0.7)
axes[2].set_ylabel('R2% Runup (m)')
axes[2].set_title('2% Exceedance Runup (Stockdon et al. 2006)')
axes[2].grid(True, alpha=0.3)

# Plot Water Level and TWL
axes[3].plot(df_merged.index, df_merged['water_level'], 'b-', linewidth=0.5, alpha=0.5, label='Water Level')
axes[3].plot(df_merged.index, df_merged['TWL'], 'k-', linewidth=0.5, alpha=0.7, label='Total Water Level (TWL)')
axes[3].axhline(y=berm_crest, color='r', linestyle='--', linewidth=2, 
                label=f'Berm Crest = {berm_crest} m NAVD88')
axes[3].set_ylabel('Elevation (m NAVD88)')
axes[3].set_title('Water Level and Total Water Level')
axes[3].legend(loc='upper right')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [13]:
# Exceedence
exceedance_mask = df_merged['TWL'] > berm_crest
percent_exceedance = exceedance_mask.sum() / len(df_merged) * 100

print(f"Berm crest elevation: {berm_crest} m NAVD88")
print(f"Total hours analyzed: {len(df_merged):,}")
print(f"Hours TWL > berm crest: {exceedance_mask.sum():,}")
print(f"Percent exceedance: {percent_exceedance:.2f}%")

# Calculate annual exceedance rates
df_merged['year'] = df_merged.index.year
annual_exceedance = df_merged.groupby('year')['TWL'].apply(
    lambda x: (x > berm_crest).sum() / len(x) * 100
)

print("\nAnnual exceedance percentages:")
for year in annual_exceedance.index:
    print(f"  {year}: {annual_exceedance[year]:.2f}%")

In [14]:
df_merged

In [15]:
import pandas as pd
import numpy as np

# =========================
# USER INPUT
# =========================
berm_elevation = 2
tp_col = "Tp"   # change if your WIS period column has a different name

# =========================
# BUILD DATAFRAME
# =========================
df = pd.DataFrame({
    "Time": pd.to_datetime(df_merged.index),
    "TWL": pd.to_numeric(df_merged["TWL"], errors="coerce")
})

# Add Tp safely
if tp_col in df_merged.columns:
    df["Tp"] = pd.to_numeric(df_merged[tp_col], errors="coerce")
else:
    df["Tp"] = np.nan

df = df.sort_values("Time").reset_index(drop=True)

# =========================
# CLEAN / FILL SMALL Tp GAPS
# =========================
# Fill small internal gaps first
df["Tp"] = df["Tp"].interpolate(limit=6)

# Then forward/backward fill any remaining gaps
df["Tp"] = df["Tp"].ffill().bfill()

# Optional debug
print("Total rows:", len(df))
print("Non-NaN TWL:", df["TWL"].notna().sum())
print("Non-NaN Tp:", df["Tp"].notna().sum())
print("NaN Tp:", df["Tp"].isna().sum())

# =========================
# FIND STORMS (TWL > berm)
# =========================
df["AboveBerm"] = df["TWL"] > berm_elevation

df["StormStart"] = df["AboveBerm"] & (~df["AboveBerm"].shift(1, fill_value=False))

df["StormID"] = df["StormStart"].cumsum()
df.loc[~df["AboveBerm"], "StormID"] = np.nan

# =========================
# BUILD STORMS
# =========================
storms = []

storm_groups = df.dropna(subset=["StormID"]).groupby("StormID")

for sid, group in storm_groups:

    group = group.sort_values("Time").copy()

    start_time = group["Time"].iloc[0]
    end_time   = group["Time"].iloc[-1]

    # Duration (hours)
    if len(group) > 1:
        dt_hours = (group["Time"].iloc[1] - group["Time"].iloc[0]).total_seconds() / 3600.0
        duration = (group["Time"].iloc[-1] - group["Time"].iloc[0]).total_seconds() / 3600.0 + dt_hours
    else:
        duration = 1.0

    # Rhigh and Rlow from TWL during the storm
    rhigh = group["TWL"].max()
    rlow  = group["TWL"].min()

    # Period from Tp at PEAK TWL
    if group["Tp"].notna().any():
        peak_idx = group["TWL"].idxmax()
        period = group.loc[peak_idx, "Tp"]
    else:
        # fallback in case Tp is still missing
        period = df["Tp"].mean()

    storms.append({
        "calendar_year": start_time.year,
        "StartTime": start_time,
        "EndTime": end_time,
        "Rhigh": rhigh,
        "Rlow": rlow,
        "Period": period,
        "duration": duration
    })

storms_df = pd.DataFrame(storms)

# =========================
# CONVERT YEAR → TIME (CASCADE)
# =========================
if not storms_df.empty:
    start_year = storms_df["calendar_year"].min()
    storms_df["time"] = storms_df["calendar_year"] - start_year + 1
else:
    storms_df["time"] = pd.Series(dtype=float)

# =========================
# FINAL FORMAT
# =========================
cascade_df = storms_df[["time", "Rhigh", "Rlow", "Period", "duration"]].copy()
cascade_df = cascade_df.reset_index(drop=True)

# =========================
# OUTPUT
# =========================
print("\nStorm summary:")
print(storms_df.head())

print("\nCASCADE format:")
print(cascade_df.head())
storms_df.to_csv("storm_summary_FOR_HANNAH.csv", index=True)
cascade_df.to_csv("cascade_storms_FOR_HANNAH.csv", index=True)

In [16]:
cascade_df